In [1]:
#Import Packages
from pathlib import Path

import geopandas as gpd

#Read in Bounding Box and Produce Data Paths

#all original input files
rawdir = Path("C:/Users/luke.smith_wildernes/OneDrive/Documents/UNI_Work/Geospatial_Data_Analytics/AT3/data/raw")

#all interim files
intdir = Path("C:/Users/luke.smith_wildernes/OneDrive/Documents/UNI_Work/Geospatial_Data_Analytics/AT3/data/interim")

#all processed output files
procdir = Path("C:/Users/luke.smith_wildernes/OneDrive/Documents/UNI_Work/Geospatial_Data_Analytics/AT3/data/processed") 

bounding_box = rawdir / "bounding_box_MGA2020.shp"

import sys

# Find the main AT3 project folder
project_dir = Path.cwd().resolve().parent

# Find the src folder
src_dir = project_dir / "src"

# Add src to Python's search path
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

#Initialise QGIS in Python
from qgis.core import *
import processing 
from processing.core.Processing import Processing

qgs = QgsApplication.instance()

if qgs is None:
    qgs = QgsApplication([], False)
    qgs.initQgis()
    Processing.initialize()

In [ ]:
#Sum of Squares and Vector Norms
# Import required packages
from math import sqrt
from qgis.core import QgsRasterLayer

# Input file paths
slope_reclass = procdir / "Slope_Reclassified.tif"
veg_reclass = procdir / "TASVEG_Reclassified_1m.tif"
watercourse_reclass = procdir / "Watercourse_Reclassified.tif"
tracks_reclass = procdir / "Tracks_Reclassified.tif"

# Load raster layers
slope = QgsRasterLayer(
    str(slope_reclass),
    "slope"
)

vegetation = QgsRasterLayer(
    str(veg_reclass),
    "vegetation"
)

watercourse = QgsRasterLayer(
    str(watercourse_reclass),
    "watercourse"
)

tracks = QgsRasterLayer(
    str(tracks_reclass),
    "tracks"
)

# Check that all rasters loaded correctly
if not slope.isValid():
    raise ValueError("Slope raster could not be loaded")

if not vegetation.isValid():
    raise ValueError("Vegetation raster could not be loaded")

if not watercourse.isValid():
    raise ValueError("Watercourse raster could not be loaded")

if not tracks.isValid():
    raise ValueError("Existing tracks raster could not be loaded")

# Calculate raster statistics
slope_stats = slope.dataProvider().bandStatistics(1)
vegetation_stats = vegetation.dataProvider().bandStatistics(1)
watercourse_stats = watercourse.dataProvider().bandStatistics(1)
tracks_stats = tracks.dataProvider().bandStatistics(1)

# Calculate sum of squared cell values
slope_sum_squares = slope_stats.sumOfSquares
vegetation_sum_squares = vegetation_stats.sumOfSquares
watercourse_sum_squares = watercourse_stats.sumOfSquares
tracks_sum_squares = tracks_stats.sumOfSquares

# Calculate vector norms
# Square root of the sum of squared cell values
slope_norm = sqrt(slope_sum_squares)
vegetation_norm = sqrt(vegetation_sum_squares)
watercourse_norm = sqrt(watercourse_sum_squares)
tracks_norm = sqrt(tracks_sum_squares)

# Print sum of squares
print("Slope sum of squares:", slope_sum_squares)
print("Vegetation sum of squares:", vegetation_sum_squares)
print("Watercourse sum of squares:", watercourse_sum_squares)
print("Existing tracks sum of squares:", tracks_sum_squares)

# Print vector norms
print("\nSlope norm:", slope_norm)
print("Vegetation norm:", vegetation_norm)
print("Watercourse norm:", watercourse_norm)
print("Existing tracks norm:", tracks_norm)


Slope sum of squares: 16121385.896135457
Vegetation sum of squares: 3090316.924104915
Watercourse sum of squares: 9577271.828687463
Existing tracks sum of squares: 18056877.282993346

Slope norm: 4015.1445672771806
Vegetation norm: 1757.9297267254215
Watercourse norm: 3094.716760656371
Existing tracks norm: 4249.338452393896


In [3]:
#Positive and Negative Ideals

# Calculate positive and negative ideal solutions for TOPSIS

# Slope
slope_positive = slope_stats.maximumValue / slope_norm
slope_negative = slope_stats.minimumValue / slope_norm

# Vegetation
vegetation_positive = vegetation_stats.maximumValue / vegetation_norm
vegetation_negative = vegetation_stats.minimumValue / vegetation_norm

# Watercourse proximity
watercourse_positive = watercourse_stats.maximumValue / watercourse_norm
watercourse_negative = watercourse_stats.minimumValue / watercourse_norm

# Existing track proximity
tracks_positive = tracks_stats.maximumValue / tracks_norm
tracks_negative = tracks_stats.minimumValue / tracks_norm

# Print results
print("Slope:", slope_positive, slope_negative)
print("Vegetation:", vegetation_positive, vegetation_negative)
print("Watercourse:", watercourse_positive, watercourse_negative)
print("Existing tracks:", tracks_positive, tracks_negative)


Slope: 0.0012452851737268047 0.00024905703474536095
Vegetation: 0.0022754038111927194 0.0005688509527981798
Watercourse: 0.0016156567423441781 0.0003231313484688356
Existing tracks: 0.0011766537441100305 0.0002353307488220061


In [6]:
# Weighted Euclidean Distance from Positive and Negative Ideals

# Define TOPSIS weights
slope_weight = 0.25
vegetation_weight = 0.45
watercourse_weight = 0.20
tracks_weight = 0.10

# Use existing reclassified raster layers:
# slope, vegetation, watercourse, tracks

# Output files
positive_distance = procdir / "TOPSIS_Positive_Distance.tif"
negative_distance = procdir / "TOPSIS_Negative_Distance.tif"

# Weighted Euclidean distance from the positive ideal
positive_expression = f"""
sqrt(
    ({slope_weight} * ("slope@1" / {slope_norm} - {slope_positive})) ^ 2 +
    ({vegetation_weight} * ("vegetation@1" / {vegetation_norm} - {vegetation_positive})) ^ 2 +
    ({watercourse_weight} * ("watercourse@1" / {watercourse_norm} - {watercourse_positive})) ^ 2 +
    ({tracks_weight} * ("tracks@1" / {tracks_norm} - {tracks_positive})) ^ 2
)
"""

# Weighted Euclidean distance from the negative ideal
negative_expression = f"""
sqrt(
    ({slope_weight} * ("slope@1" / {slope_norm} - {slope_negative})) ^ 2 +
    ({vegetation_weight} * ("vegetation@1" / {vegetation_norm} - {vegetation_negative})) ^ 2 +
    ({watercourse_weight} * ("watercourse@1" / {watercourse_norm} - {watercourse_negative})) ^ 2 +
    ({tracks_weight} * ("tracks@1" / {tracks_norm} - {tracks_negative})) ^ 2
)
"""

# Calculate positive ideal distance raster
processing.run(
    "native:rastercalc",
    {
        "LAYERS": [
            slope,
            vegetation,
            watercourse,
            tracks
        ],
        "EXPRESSION": positive_expression,
        "EXTENT": slope.extent(),
        "CELL_SIZE": slope.rasterUnitsPerPixelX(),
        "CRS": slope.crs(),
        "OUTPUT": str(positive_distance)
    }
)

# Calculate negative ideal distance raster
processing.run(
    "native:rastercalc",
    {
        "LAYERS": [
            slope,
            vegetation,
            watercourse,
            tracks
        ],
        "EXPRESSION": negative_expression,
        "EXTENT": slope.extent(),
        "CELL_SIZE": slope.rasterUnitsPerPixelX(),
        "CRS": slope.crs(),
        "OUTPUT": str(negative_distance)
    }
)

print("Positive ideal distance:", positive_distance)
print("Negative ideal distance:", negative_distance)



Positive ideal distance: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\data\processed\TOPSIS_Positive_Distance.tif
Negative ideal distance: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\data\processed\TOPSIS_Negative_Distance.tif


In [7]:
#Calculate TOPSIS closeness coefficient and topsis result 

# Load positive and negative ideal distance rasters
positive_distance_layer = QgsRasterLayer(
    str(positive_distance),
    "positive_distance"
)

negative_distance_layer = QgsRasterLayer(
    str(negative_distance),
    "negative_distance"
)

# Check that both rasters loaded correctly
if not positive_distance_layer.isValid():
    raise ValueError("Positive distance raster could not be loaded")

if not negative_distance_layer.isValid():
    raise ValueError("Negative distance raster could not be loaded")

# Output TOPSIS suitability raster
topsis_output = procdir / "TOPSIS_Walking_Track_Suitability.tif"

# Calculate relative closeness to the positive ideal
topsis_expression = (
    '"negative_distance@1" / '
    '("positive_distance@1" + "negative_distance@1")'
)

# Run raster calculator
processing.run(
    "native:rastercalc",
    {
        "LAYERS": [
            positive_distance_layer,
            negative_distance_layer
        ],
        "EXPRESSION": topsis_expression,
        "EXTENT": positive_distance_layer.extent(),
        "CELL_SIZE": positive_distance_layer.rasterUnitsPerPixelX(),
        "CRS": positive_distance_layer.crs(),
        "OUTPUT": str(topsis_output)
    }
)

print("TOPSIS suitability raster created:", topsis_output)



TOPSIS suitability raster created: C:\Users\luke.smith_wildernes\OneDrive\Documents\UNI_Work\Geospatial_Data_Analytics\AT3\data\processed\TOPSIS_Walking_Track_Suitability.tif
